In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("cleaned_sales.csv")

print("Dataset Loaded Successfully!")
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

display(df.head())

Dataset Loaded Successfully!
Total Rows: 1003
Total Columns: 18


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales,Duplicate_Order_ID,Order_Year,Order_Month,Order_Day,Age_Group,Sales_Outlier
0,ORD100002,25-02-2025,CUST5529,Customer_227,30.0,Female,Bengaluru,Rice,Grocery,7.0,2829.77,19808.39,False,2025.0,2.0,25.0,Adult,False
1,ORD100003,14-10-2025,CUST3127,Customer_182,63.0,Male,Bengaluru,Book,Education,5.0,27906.16,139530.80,False,2025.0,10.0,14.0,Senior,False
2,ORD100004,13-05-2025,CUST8887,Customer_487,62.0,Female,Bengaluru,Book,Education,8.0,37491.06,299928.48,False,2025.0,5.0,13.0,Senior,False
3,ORD100005,02-12-2025,CUST2515,Customer_470,65.0,Female,Kolkata,Mobile,Electronics,9.0,28541.36,256872.24,False,2025.0,12.0,2.0,Senior,False
4,ORD100006,20-11-2025,CUST4796,Customer_380,44.0,Male,Bengaluru,Rice,Grocery,10.0,14036.59,140365.90,False,2025.0,11.0,20.0,Middle_Aged,False


In [2]:
# Check required columns

print("Available Columns:")
print(df.columns.tolist())

print("\nGender Distribution:")
print(df["Gender"].value_counts(dropna=False))

print("\nMissing Values:")
print(df[["Gender", "Total_Sales"]].isnull().sum())

print("\nSales Data Summary:")
display(df.groupby("Gender")["Total_Sales"].describe())

Available Columns:
['Order_ID', 'Order_Date', 'Customer_ID', 'Customer_Name', 'Age', 'Gender', 'City', 'Product', 'Category', 'Quantity', 'Unit_Price', 'Total_Sales', 'Duplicate_Order_ID', 'Order_Year', 'Order_Month', 'Order_Day', 'Age_Group', 'Sales_Outlier']

Gender Distribution:
Gender
Male      511
Female    489
NaN         3
Name: count, dtype: int64

Missing Values:
Gender         3
Total_Sales    3
dtype: int64

Sales Data Summary:


,count,mean,std,min,25%,50%,75%,max
Gender,,,,,,,,
Female,489.0,136883.208630,111788.031275,1005.21,45270.450,108470.60,197341.68,490866.4
Male,511.0,141807.339785,116327.975385,437.34,47543.825,109237.86,218334.38,493677.5


In [3]:
# Create separate dataset for hypothesis testing

test_df = df[["Gender", "Total_Sales"]].copy()

# Remove missing values
test_df = test_df.dropna(subset=["Gender", "Total_Sales"])
# dropna ka matlab hai ki hum un rows ko hata rahe hain jisme Gender ya Total_Sales ka data missing hai. 
# Isse hum ensure karte hain ki humare analysis ke liye sirf complete records hi use ho rahe hain.
# Keep only Male and Female records
test_df = test_df[
    test_df["Gender"].isin(["Male", "Female"])
]

print("Testing Dataset Ready!")
print("Total Records:", len(test_df))

print("\nGender Distribution:")
print(test_df["Gender"].value_counts())

print("\nMissing Values:")
print(test_df.isnull().sum())

Testing Dataset Ready!
Total Records: 1000

Gender Distribution:
Gender
Male      511
Female    489
Name: count, dtype: int64

Missing Values:
Gender         0
Total_Sales    0
dtype: int64


In [4]:
# Define Hypotheses

H0 = "Male and Female customers have equal average sales."

H1 = "Male and Female customers have different average sales."

print("BUSINESS QUESTION:")
print("Is there a significant difference in average sales between Male and Female customers?")

print("\nNULL HYPOTHESIS (H0):")
print(H0)

print("\nALTERNATIVE HYPOTHESIS (H1):")
print(H1)

alpha = 0.05

print("\nSignificance Level:", alpha)

BUSINESS QUESTION:
Is there a significant difference in average sales between Male and Female customers?

NULL HYPOTHESIS (H0):
Male and Female customers have equal average sales.

ALTERNATIVE HYPOTHESIS (H1):
Male and Female customers have different average sales.

Significance Level: 0.05


In [5]:
# Separate Male and Female Sales

male_sales = test_df[             # iska matlab hai ki hum test_df dataframe me se un rows ko select kar rahe hain jisme Gender column ka value "Male" hai, 
    test_df["Gender"] == "Male"   # aur phir un rows ke Total_Sales column ke values ko male_sales variable me store kar rahe hain.
    
]["Total_Sales"]

female_sales = test_df[
    test_df["Gender"] == "Female"  # iska matlab hai ki hum test_df dataframe me se un rows ko select kar rahe hain jisme Gender column ka value "Female" hai, 
]["Total_Sales"  ]           # aur phir un rows ke Total_Sales column ke values ko female_sales variable me store kar rahe hain.




# Perform Welch's Independent T-Test

t_stat, p_value = stats.ttest_ind(
    male_sales,
    female_sales,
    equal_var=False
)

print("WELCH'S INDEPENDENT T-TEST RESULTS")
print("-----------------------------------")

print("Male Customers:", len(male_sales))
print("Female Customers:", len(female_sales))

print("Male Average Sales:", male_sales.mean())
print("Female Average Sales:", female_sales.mean())

print("T-Statistic:", t_stat) # t_stat ka matlab hai ki humare observed data ke basis par null hypothesis ke true hone ki probability kitni hai.
print("P-Value:", p_value) # p_value ka matlab hai ki humare observed data ke basis par null hypothesis ke true hone ki probability kitni hai.
#Agar p_value chhoti hai (alpha se chhoti), to hum null hypothesis ko reject karte hain aur alternative hypothesis ko accept karte hain.
# Agar p_value badi hai, to hum null hypothesis ko reject nahi karte hain.

# Decision

if p_value < alpha:
    print("\nDecision: Reject H0")
    print("There is a statistically significant difference.")
else:
    print("\nDecision: Fail to Reject H0")
    print("No statistically significant difference was found.")

WELCH'S INDEPENDENT T-TEST RESULTS
-----------------------------------
Male Customers: 511
Female Customers: 489
Male Average Sales: 141807.3397847358
Female Average Sales: 136883.20862985688
T-Statistic: 0.6826103913380815
P-Value: 0.4950114785145835

Decision: Fail to Reject H0
No statistically significant difference was found.


In [6]:
# Calculate 95% Confidence Interval using Welch's method

male_mean = male_sales.mean()
female_mean = female_sales.mean()

male_var = male_sales.var(ddof=1)
female_var = female_sales.var(ddof=1)

n1 = len(male_sales)
n2 = len(female_sales)

# Difference between means
mean_difference = male_mean - female_mean

# Standard Error
se = np.sqrt((male_var / n1) + (female_var / n2))

# Welch-Satterthwaite degrees of freedom
df_welch = ((male_var/n1 + female_var/n2)**2) / (
    ((male_var/n1)**2 / (n1-1)) +
    ((female_var/n2)**2 / (n2-1))
)

# 95% Confidence Interval
t_critical = stats.t.ppf(0.975, df_welch)

lower = mean_difference - t_critical * se
upper = mean_difference + t_critical * se

print("95% CONFIDENCE INTERVAL")
print("-----------------------")
print("Mean Difference (Male - Female):", round(mean_difference, 2))
print("Lower Limit:", round(lower, 2))
print("Upper Limit:", round(upper, 2))
print("Confidence Level: 95%")

95% CONFIDENCE INTERVAL
-----------------------
Mean Difference (Male - Female): 4924.13
Lower Limit: -9231.58
Upper Limit: 19079.85
Confidence Level: 95%


In [8]:
summary = pd.DataFrame({
    "Metric": [
        "Male Orders",
        "Female Orders",
        "Male Average Sales",
        "Female Average Sales",
        "Mean Difference",
        "T-Statistic",
        "P-Value",
        "Confidence Level",
        "CI Lower Limit",
        "CI Upper Limit",
        "Conclusion"
    ],
    "Result": [
        len(male_sales),
        len(female_sales),
        round(male_mean, 2),
        round(female_mean, 2),
        round(mean_difference, 2),
        round(t_stat, 4),
        round(p_value, 4),
        "95%",
        round(lower, 2),
        round(upper, 2),
        "No statistically significant difference"
    ]
})

summary.to_csv(
    "Hypothesis_Testing_Summary.csv",
    index=False
)

display(summary)

print("Hypothesis Testing Summary Exported Successfully!")

,Metric,Result
0,Male Orders,511
1,Female Orders,489
2,Male Average Sales,141807.34
3,Female Average Sales,136883.21
4,Mean Difference,4924.13
5,T-Statistic,0.6826
6,P-Value,0.495
7,Confidence Level,95%
8,CI Lower Limit,-9231.58
9,CI Upper Limit,19079.85


Hypothesis Testing Summary Exported Successfully!
